# Download Eval Videos and pre-trained weights

## These are the pre-trained weights from research paper authors

In [ ]:
!mkdir -p Trained_Models
!pip install gdown --quiet
!gdown "1aWLukhsRV5Fi_DFJUbL5nBdwNhGDpNe0" -O Trained_Models/MCUCoder.pth

# Download Testing Video

In [ ]:
!mkdir -p Eval_Videos
!curl -o Eval_Videos/input_video.mp4 "https://www.w3schools.com/html/mov_bbb.mp4"

# Check NVIDIA GPU

In [ ]:
!nvidia-smi

# Checking and Installing Packages

In [ ]:
# Run this cell once before anything else:

!pip install "torch>=2.3.1" "torchvision>=0.18.1" --dry-run
!pip install "lightning>=2.3.0" --dry-run
!pip install "opencv-python>=4.10.0.84" --dry-run
!pip install "compressai>=1.2.6" --dry-run
!pip install "dahuffman>=0.4.1" --dry-run
!pip install "wandb>=0.17.2" --dry-run
!pip install "torchsummary>=1.5.1" --dry-run
!pip install "seaborn>=0.13.2" --dry-run
!pip install "tensorflow>=2.18.0" --dry-run

In [ ]:
# Run this cell once before anything else:

# Already satisfied:
# !pip install "torch>=2.3.1" "torchvision>=0.18.1"
# !pip install "opencv-python>=4.10.0.84"
# !pip install "wandb>=0.17.2"
# !pip install "torchsummary>=1.5.1"
# !pip install "seaborn>=0.13.2"
# !pip install "tensorflow>=2.18.0"

# Needs to be installed:
!pip install "lightning>=2.3.0"
!pip install "compressai>=1.2.6"
!pip install "dahuffman>=0.4.1"

# Imports

In [ ]:
import os
import cv2
import json
import math
import pickle
import random

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision import datasets
from torch import Tensor
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.sampler import SubsetRandomSampler

import lightning as L
from lightning.pytorch.loggers import WandbLogger
from lightning.pytorch.callbacks import LearningRateMonitor

import wandb
from dahuffman import HuffmanCodec
from PIL import Image
from tqdm import tqdm

from torchmetrics.image import MultiScaleStructuralSimilarityIndexMeasure
from compressai.layers import (
  GDN,
  AttentionBlock,
  ResidualBlock,
  ResidualBlockUpsample,
  ResidualBlockWithStride,
  conv3x3,
  conv1x1,
  subpel_conv3x3,
)
from compressai.models.utils import conv, deconv

# Plot settings
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
sns.set(font_scale=1.2)
plt.rc('legend', fontsize=10)
sns.set_style("whitegrid")

device: str = None

if torch.backends.mps.is_available():
  device = 'mps'
elif torch.cuda.is_available():
  device = 'cuda'
else:
  device = 'cpu'

print(f"Using device: {device}")

# Config

In [ ]:
import os
import sys

# Determine the project root directory based on the execution environment
try:
  # If running as a standard .py script
  PROJECT_ROOT = os.path.dirname(os.path.abspath(__file__))
except NameError:
  # If running in a Jupyter Notebook (__file__ is not defined)
  PROJECT_ROOT = os.path.abspath(os.getcwd())

# Add the project root to sys.path so custom modules can be imported
if PROJECT_ROOT not in sys.path:
  sys.path.append(PROJECT_ROOT)

CONFIG = {
  # --- Training ---
  "train_data_dir":   os.path.join(PROJECT_ROOT, "high_res_imagenet"),   # Path to training images
  "val_data_dir":     os.path.join(PROJECT_ROOT, "KODAK"),               # Path to validation images
  "batch_size":       16,
  "number_of_iterations": 1_000_000,
  "number_of_channels":   196,
  "loss":             "msssim",   # "msssim" or "mse"
  "wandb_name":       "MCUCoderRL",
  "wandb_project":    "MCUCoderRL",
  "model_save_path":  os.path.join(PROJECT_ROOT, "MCUCoder.pth"),

  # --- ImageNet Preparation ---
  # Assuming you move the raw ImageNet data into a 'datasets/imagenet/train' folder in your project root.
  # If it remains on a separate drive, you can replace this with the hardcoded absolute path.
  "imagenet_raw_dir": os.path.join(PROJECT_ROOT, "datasets", "imagenet", "train"),
  "imagenet_out_dir": os.path.join(PROJECT_ROOT, "high_res_imagenet"),
  "num_images_to_select": 300_000,

  # --- Video Encoding/Decoding ---
  "model_path":       os.path.join(PROJECT_ROOT, "Trained_Models", "MCUCoder.pth"),       # Path to trained .pth file
  "video_path":       os.path.join(PROJECT_ROOT, "Eval_Videos" ,"input_video.mp4"),    # Path to input video
  "output_dir":       os.path.join(PROJECT_ROOT, "outputs"),            # Directory for output files
  "eval_batch_size":  1024,
}

# Data Preprocessing

In [ ]:
class CustomImageDataset(Dataset):
  """Loads all image files from a flat directory."""

  def __init__(self, root_dir, transform=None):
    self.root_dir = root_dir
    self.transform = transform
    self.image_files = [
      f for f in os.listdir(root_dir)
      if os.path.isfile(os.path.join(root_dir, f))
    ]

  def __len__(self):
    return len(self.image_files)

  def __getitem__(self, idx):
    img_name = os.path.join(self.root_dir, self.image_files[idx])
    image = Image.open(img_name)
    if self.transform:
      image = self.transform(image)
    return image

# Model Definition

In [ ]:
class ResidualBottleneckBlock(nn.Module):
  """Residual bottleneck block (He et al., CVPR 2016).

  Sandwiches a 3x3 conv between two 1x1 convolutions to reduce parameters.

  Args:
    in_ch (int): Number of input channels.
    out_ch (int): Number of output channels.
  """

  def __init__(self, in_ch: int, out_ch: int):
    super().__init__()
    mid_ch = min(in_ch, out_ch) // 2
    self.conv1 = conv1x1(in_ch, mid_ch)
    self.relu1 = nn.ReLU(inplace=True)
    self.conv2 = conv3x3(mid_ch, mid_ch)
    self.relu2 = nn.ReLU(inplace=True)
    self.conv3 = conv1x1(mid_ch, out_ch)
    self.skip  = conv1x1(in_ch, out_ch) if in_ch != out_ch else nn.Identity()

  def forward(self, x: Tensor) -> Tensor:
    identity = self.skip(x)
    out = self.relu1(self.conv1(x))
    out = self.relu2(self.conv2(out))
    out = self.conv3(out)
    return out + identity


def psnr_batch(img1, img2):
  """Compute mean PSNR over a batch (returns a scalar tensor)."""
  mse = F.mse_loss(img1, img2, reduction='none')
  mse = mse.view(mse.size(0), -1).mean(dim=1)
  psnr_values = 20 * torch.log10(1.0 / torch.sqrt(mse))
  return torch.mean(psnr_values.detach().cpu())


def ms_ssim_to_db(ms_ssim_val):
  """Convert MS-SSIM to dB."""
  return -10 * np.log10(1 - ms_ssim_val)


def ms_ssim_batch(img1, img2, data_range=1.0):
  """Compute per-image MS-SSIM (dB) over a batch."""
  ms_ssim_metric = MultiScaleStructuralSimilarityIndexMeasure(
    data_range=data_range
  ).to(device)
  ms_ssim_values = [
    ms_ssim_metric(img1[i].unsqueeze(0), img2[i].unsqueeze(0)).item()
    for i in range(img1.size(0))
  ]
  return torch.tensor(
    [ms_ssim_to_db(v) for v in ms_ssim_values], device=device
  )


class Encoder(L.LightningModule):
  """Lightweight convolutional encoder."""

  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(3,  16, 7, stride=2, padding=3)
    self.conv2 = nn.Conv2d(16, 16, 5, stride=4, padding=1)
    self.conv3 = nn.Conv2d(16, 12, 3, stride=1, padding=1)

  def forward(self, x):
    x = F.relu(self.conv1(x))
    x = F.relu(self.conv2(x))
    return self.conv3(x)


class Decoder(L.LightningModule):
  """Deep residual decoder with attention blocks."""

  def __init__(self, N=196):
    super().__init__()
    self.N = N
    self.dec = nn.Sequential(
      AttentionBlock(12),
      deconv(12, N, kernel_size=5, stride=2),

      ResidualBottleneckBlock(N, N),
      ResidualBottleneckBlock(N, N),
      ResidualBottleneckBlock(N, N),
      AttentionBlock(N),
      ResidualBottleneckBlock(N, N),
      ResidualBottleneckBlock(N, N),
      ResidualBottleneckBlock(N, N),
      AttentionBlock(N),
      ResidualBottleneckBlock(N, N),
      ResidualBottleneckBlock(N, N),
      ResidualBottleneckBlock(N, N),
      deconv(N, N, kernel_size=5, stride=2),

      AttentionBlock(N),
      ResidualBottleneckBlock(N, N),
      ResidualBottleneckBlock(N, N),
      ResidualBottleneckBlock(N, N),
      AttentionBlock(N),
      ResidualBottleneckBlock(N, N),
      ResidualBottleneckBlock(N, N),
      ResidualBottleneckBlock(N, N),
      AttentionBlock(N),
      ResidualBottleneckBlock(N, N),
      ResidualBottleneckBlock(N, N),
      ResidualBottleneckBlock(N, N),
      deconv(N, 3, kernel_size=5, stride=2),
    )

  def forward(self, x):
    return torch.sigmoid(self.dec(x))


class MCUCoder(L.LightningModule):
  """Rate-less image compression model (encoder + decoder)."""

  def __init__(self, loss=None, N=196):
    super().__init__()
    self.encoder   = Encoder()
    self.decoder   = Decoder(N=N)
    self.loss_func = loss
    self.ms_ssim   = MultiScaleStructuralSimilarityIndexMeasure(data_range=1.0)

    self.p             = None
    self.replace_value = 0

    self.training_step_loss      = []
    self.validation_step_psnr    = []
    self.validation_step_ms_ssim = []
    self.saved_images            = []

  # ------------------------------------------------------------------
  def rate_less(self, x):
    """Zero-out the tail channels according to the rate parameter p."""
    temp_x = x.clone()
    p = self.p if self.p else np.random.randint(1, 13) / 12
    if p != 1.0:
      keep = int(p * x.shape[1])
      fill = torch.full(
        (x.shape[0], x.shape[1] - keep, x.shape[2], x.shape[3]),
        self.replace_value
      )
      temp_x[:, -fill.shape[1]:, :, :] = fill
    return temp_x

  def forward(self, x):
    x = self.encoder(x)
    x = self.rate_less(x)
    if not self.training:
      noise = torch.rand_like(x) * 0.02 - 0.01
      x = x + noise
    x = self.decoder(x)
    self.rec_image = x.detach().clone()
    return x

  # ------------------------------------------------------------------
  def configure_optimizers(self):
    optimizer    = torch.optim.Adam(self.parameters(), lr=1e-4, betas=(0.9, 0.999))
    first_phase  = int(self.trainer.max_steps * 0.95)
    scheduler    = torch.optim.lr_scheduler.StepLR(
      optimizer, step_size=first_phase, gamma=0.1
    )
    return [optimizer], [{'scheduler': scheduler, 'interval': 'step'}]

  # ------------------------------------------------------------------
  def training_step(self, train_batch, batch_idx):
    images  = train_batch
    outputs = self(images)

    if self.loss_func == 'msssim':
      loss = 1 - self.ms_ssim(outputs, images)
      self.log('during_train_loss_ms_ssim', loss, on_epoch=True,
           prog_bar=True, logger=True)
    elif self.loss_func == 'mse':
      loss = nn.MSELoss()(outputs, images)
      self.log('during_train_loss_mse', loss, on_epoch=True,
           prog_bar=True, logger=True)

    self.training_step_loss.append(loss)
    return {'loss': loss}

  def on_train_epoch_end(self):
    loss = torch.stack(self.training_step_loss).mean()
    self.log('train_loss_epoch', loss, on_epoch=True, prog_bar=True, logger=True)
    self.training_step_loss.clear()
    lr = self.trainer.lr_scheduler_configs[0].scheduler.get_last_lr()[0]
    self.log('learning_rate', lr, on_step=False, on_epoch=True)

  # ------------------------------------------------------------------
  def validation_step(self, val_batch, batch_idx):
    self.saved_images = []
    msssim_temp, psnr_temp = [], []

    for t in [2, 6, 12]:
      self.p  = t / 12
      images  = val_batch
      outputs = self(images)

      psnr_temp.append(psnr_batch(outputs, images))
      msssim_temp.append(ms_ssim_batch(outputs, images))
      self.saved_images.append(self.rec_image[0])

    self.validation_step_psnr.append(psnr_temp)
    self.validation_step_ms_ssim.append(msssim_temp)
    self.p = None
    return {'loss': psnr_temp[-1], 'ms_ssim': msssim_temp[-1]}

  def on_validation_epoch_end(self):
    for i, suffix in enumerate(['2l', '6l', '12l']):
      psnr = torch.stack([x[i] for x in self.validation_step_psnr]).mean()
      self.log(f'val_psnr_{suffix}_epoch', psnr, on_epoch=True,
           prog_bar=True, logger=True)
      ms_ssim = torch.stack([x[i] for x in self.validation_step_ms_ssim]).mean()
      self.log(f'val_ms_ssim_{suffix}_epoch', ms_ssim, on_epoch=True,
           prog_bar=True, logger=True)

    self.logger.experiment.log({"rec_image_2l":  wandb.Image(self.saved_images[0])})
    self.logger.experiment.log({"rec_image_6l":  wandb.Image(self.saved_images[1])})
    self.logger.experiment.log({"rec_image_12l": wandb.Image(self.saved_images[2])})

    self.p = None
    self.validation_step_psnr.clear()
    self.validation_step_ms_ssim.clear()

# Training

In [ ]:
def set_seeds():
  torch.manual_seed(0)
  random.seed(10)
  np.random.seed(0)


def load_datasets(train_dir, val_dir, crop=224):
  train_transform = transforms.Compose([
    transforms.Resize(crop),
    transforms.CenterCrop(crop),
    transforms.ToTensor(),
  ])
  val_transform = transforms.Compose([
    transforms.Resize(crop),
    transforms.CenterCrop(crop),
    transforms.ToTensor(),
  ])
  return (
    CustomImageDataset(root_dir=train_dir, transform=train_transform),
    CustomImageDataset(root_dir=val_dir,   transform=val_transform),
  )


def run_training(cfg=CONFIG):
  """Launch training using settings in CONFIG dict."""
  wandb.require("core")
  set_seeds()

  train_ds, val_ds = load_datasets(cfg["train_data_dir"], cfg["val_data_dir"])

  model        = MCUCoder(loss=cfg["loss"], N=cfg["number_of_channels"]).to(device)
  wandb_logger = WandbLogger(name=cfg["wandb_name"], project=cfg["wandb_project"])

  train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], num_workers=2)
  val_loader   = DataLoader(val_ds,   batch_size=2,                 num_workers=2)

  trainer = L.Trainer(
    accelerator="mps" if device == "mps" else ("gpu" if device == "cuda" else "cpu"),
    max_steps=cfg["number_of_iterations"],
    logger=wandb_logger,
    enable_checkpointing=False,
  )
  trainer.fit(model, train_loader, val_loader)
  torch.save(model.state_dict(), cfg["model_save_path"])
  print(f"Training complete. Model saved to {cfg['model_save_path']}")
  return model

In [ ]:
# Uncomment to train:
# model = run_training(CONFIG)

# ImageNet Preparation

In [ ]:
def get_image_resolutions(image_folder):
  """Walk a directory tree and return (path, width*height) for each image."""
  resolutions = []
  for root, dirs, files in tqdm(os.walk(image_folder)):
    for file in files:
      if file.lower().endswith(('.png', '.jpg', '.jpeg')):
        image_path = os.path.join(root, file)
        try:
          with Image.open(image_path) as img:
            w, h = img.size
            resolutions.append((image_path, w * h))
        except Exception as e:
          print(f"Error opening {image_path}: {e}")
  return resolutions


def select_largest_images(resolutions, num_images):
  """Return paths of the `num_images` highest-resolution images."""
  return [p for p, _ in sorted(resolutions, key=lambda x: x[1], reverse=True)[:num_images]]


def add_random_noise(image):
  """Add small uniform noise to a PIL image."""
  arr = np.array(image)
  noise = np.random.uniform(0, 1, arr.shape).astype(np.uint8)
  return Image.fromarray(np.clip(arr + noise, 0, 255).astype(np.uint8))


def resize_image(image, width, height):
  """Halve the resolution if the smaller side exceeds 512 px."""
  if min(width, height) > 512:
    arr = cv2.resize(
      np.array(image),
      (width // 2, height // 2),
      interpolation=cv2.INTER_CUBIC
    )
    return Image.fromarray(arr)
  return image


def save_selected_images(image_paths, output_folder):
  """Copy selected images (with optional resize + noise) to output_folder as PNG."""
  os.makedirs(output_folder, exist_ok=True)
  for image_path in tqdm(image_paths, desc="Saving images"):
    try:
      name = os.path.splitext(os.path.basename(image_path))[0]
      out  = os.path.join(output_folder, f"{name}.png")
      with Image.open(image_path) as img:
        w, h = img.size
        img  = resize_image(img, w, h)
        img  = add_random_noise(img)
        img.convert("RGB").save(out, format="PNG")
    except Exception as e:
      print(f"Error saving {image_path}: {e}")


def prepare_imagenet(cfg=CONFIG):
  """Select the largest images from an ImageNet folder and save them."""
  print("Getting image resolutions...")
  resolutions = get_image_resolutions(cfg["imagenet_raw_dir"])
  print(f"Selecting {cfg['num_images_to_select']} largest images...")
  largest = select_largest_images(resolutions, cfg["num_images_to_select"])
  print(f"Saving to {cfg['imagenet_out_dir']} ...")
  save_selected_images(largest, cfg["imagenet_out_dir"])
  print("Done!")

In [ ]:
# Uncomment to prepare ImageNet:
# prepare_imagenet(CONFIG)

# Video Encoding / Decoding

In [ ]:
# ── Optional TFLite encoder (requires tensorflow) ──────────────────────────

def encode_TFLite(model_path, X):
  """Run the encoder via a TFLite model (quantized INT8 deployment)."""
  import tensorflow as tf
  x_data = np.copy(X.to('cpu').numpy())
  interpreter = tf.lite.Interpreter(model_path=model_path)
  interpreter.allocate_tensors()
  input_details  = interpreter.get_input_details()[0]
  output_details = interpreter.get_output_details()[0]

  input_scale, input_zero_point = input_details["quantization"]
  if (input_scale, input_zero_point) != (0.0, 0):
    x_data = (x_data / input_scale + input_zero_point).astype(input_details["dtype"])

  predictions = np.empty(
    (x_data.shape[0], 12, 28, 28), dtype=output_details["dtype"]
  )
  for i in range(len(x_data)):
    interpreter.set_tensor(input_details["index"], [x_data[i]])
    interpreter.invoke()
    predictions[i] = np.copy(interpreter.get_tensor(output_details["index"])[0])

  output_scale, output_zero_point = output_details["quantization"]
  if (output_scale, output_zero_point) != (0.0, 0):
    predictions = (predictions.astype(np.float32) - output_zero_point) * output_scale

  return torch.from_numpy(predictions).to('cuda')


# ── Video dataset ───────────────────────────────────────────────────────────

class VideoDataset(Dataset):
  """Loads all frames from a video file as PIL Images."""

  def __init__(self, video_path, transform=None):
    self.video_path = video_path
    self.transform  = transform
    self.frames     = self._load_frames()

  def _load_frames(self):
    cap    = cv2.VideoCapture(self.video_path)
    frames = []
    while cap.isOpened():
      ret, frame = cap.read()
      if not ret:
        break
      frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
    cap.release()
    return frames

  def __len__(self):
    return len(self.frames)

  def __getitem__(self, idx):
    frame = self.frames[idx]
    if self.transform:
      frame = self.transform(frame)
    return frame, idx


def get_video_frames(video_path, resize=224):
  transform = transforms.Compose([
    transforms.Resize((resize, resize), antialias=True),
    transforms.ToTensor(),
  ])
  ds = VideoDataset(video_path=video_path, transform=transform)
  return ds, len(ds)


# ── Quantization helpers ─────────────────────────────────────────────────────

def quantization(data, filter_number, codec, step=4):
  """Quantize a feature map to uint8 using per-filter min/max."""
  min_, max_ = codec['min'][filter_number], codec['max'][filter_number]
  data = ((data - min_) / (max_ - min_) * 255).type(torch.uint8)
  return (data / step).type(torch.uint8)


def quantization_and_dequantization(data, filter_number, codec, step=4):
  """Quantize then dequantize (simulates lossy round-trip)."""
  min_, max_ = codec['min'][filter_number], codec['max'][filter_number]
  data = ((data - min_) / (max_ - min_) * 255).type(torch.uint8)
  data = (data / step).type(torch.uint8) * step
  return data.float() / 255.0 * (max_ - min_) + min_


def quantization_and_huffman(data, filter_number, codec, step=4):
  """Return the Huffman-encoded size of a feature map in KB."""
  data = data.reshape(-1)
  quantized = quantization(data, filter_number, codec, step).cpu().numpy()
  encoded   = codec['codec'][filter_number].encode(quantized)
  return len(encoded) / 1024


# ── Per-batch metrics ────────────────────────────────────────────────────────

def psnr_batch_video(img1, img2):
  """Per-image PSNR (returns a list of values, not a scalar)."""
  mse = F.mse_loss(img1, img2, reduction='none')
  mse = mse.view(mse.size(0), -1).mean(dim=1)
  return 20 * torch.log10(1.0 / torch.sqrt(mse))


def ms_ssim_batch_video(img1, img2, data_range=1.0):
  """Per-image MS-SSIM in dB (returns a list)."""
  metric = MultiScaleStructuralSimilarityIndexMeasure(data_range=data_range)
  values = [
    metric(img1[i].unsqueeze(0), img2[i].unsqueeze(0)).item()
    for i in range(img1.size(0))
  ]
  return [ms_ssim_to_db(v) for v in values]


# ── Saving helpers ───────────────────────────────────────────────────────────

def save_image(img_np, path):
  """Save a (C, H, W) numpy array as a PNG image."""
  plt.imsave(path, np.transpose(img_np, (1, 2, 0)))


def save_latent_images(latent_tensor, path):
  """Save the 12 latent feature maps (12×28×28) in a 3×4 grid."""
  if latent_tensor.is_cuda:
    latent_tensor = latent_tensor.cpu()
  latent = latent_tensor.detach().numpy()
  fig, axes = plt.subplots(3, 4, figsize=(12, 9))
  for i, ax in enumerate(axes.flatten()):
    ax.imshow(latent[i], cmap='gray')
    ax.axis('off')
    ax.set_title(f'Filter {i+1}')
  plt.tight_layout()
  plt.savefig(path)
  plt.close()


# ── Codec creation ───────────────────────────────────────────────────────────

def create_codec(test_dataset, model):
  """Build per-filter min/max and Huffman codec from a calibration set."""
  codec_setting = {'min': {}, 'max': {}, 'codec': {}}
  temp_loader = DataLoader(test_dataset, batch_size=5000, num_workers=1)
  images, _ = next(iter(temp_loader))
  images = images.to(device)

  for i in tqdm(range(12), desc="Building codec"):
    encoded = model.encoder(images)
    data = encoded[:, i, :, :].reshape(-1).detach().clone()
    min_, max_ = torch.min(data), torch.max(data)
    data = ((data - min_) / (max_ - min_) * 255).type(torch.uint8)
    data = (data / 4).type(torch.uint8).cpu().numpy()
    # Ensure all possible quantization levels appear at least once
    for j in range(64):
      if j not in data:
        data = np.append(data, j)
    hcodec = HuffmanCodec.from_data(data)
    codec_setting['min'][i] = min_
    codec_setting['max'][i] = max_
    codec_setting['codec'][i] = hcodec

  del temp_loader
  return codec_setting


# ── Evaluation loop ──────────────────────────────────────────────────────────

def eval_model(model, used_filter, test_dataset, codec, output_dir, output_video_dir):
  """Decode all frames using `used_filter` channels; return size/PSNR/MS-SSIM lists."""
  size_list, psnr_list, ms_ssim_list = [], [], []
  test_loader = DataLoader(test_dataset, batch_size=1, num_workers=1)

  fourcc    = cv2.VideoWriter_fourcc(*'mp4v')
  vid_path  = os.path.join(output_video_dir, f'decode_with_{used_filter}_filters.mp4')
  vid_writer = None

  for frame_idx, (images, _) in enumerate(test_loader):
    images  = images.to(device)
    encoded = model.encoder(images)
    model.replace_value = 0
    encoded = model.rate_less(encoded)

    for j in range(used_filter):
      encoded[0, j] = quantization_and_dequantization(encoded[0, j], j, codec)

    outputs = model.decoder(encoded)

    # Initialize video writer on first frame
    if vid_writer is None:
      h, w = outputs[0].shape[1:3]
      vid_writer = cv2.VideoWriter(vid_path, fourcc, 20, (w, h))

    frame = (outputs[0].cpu().detach().numpy().transpose(1, 2, 0) * 255).astype(np.uint8)
    vid_writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))

    # Save sample frames
    if frame_idx in [0, 50, 100]:
      sample_dir = os.path.join(output_dir, "decoded_frames_samples")
      os.makedirs(sample_dir, exist_ok=True)
      save_image(
        outputs[0].cpu().detach().numpy(),
        os.path.join(sample_dir, f"frame_{frame_idx}_decoded_{used_filter}_filters.png")
      )

    # Metrics
    size_list.append(
      np.sum([quantization_and_huffman(encoded[0][fm], fm, codec)
          for fm in range(used_filter)])
    )
    psnr_list.extend(psnr_batch_video(outputs.detach().cpu(), images.detach().cpu()).tolist())
    ms_ssim_list.extend(ms_ssim_batch_video(outputs.detach().cpu(), images.detach().cpu()))

  if vid_writer:
    vid_writer.release()
  return size_list, psnr_list, ms_ssim_list


# ── Main evaluation entry point ──────────────────────────────────────────────

def run_video_eval(cfg=CONFIG):
  """Load a trained model, encode/decode a video, and plot RD curves."""
  set_seeds()

  model = MCUCoder()
  state_dict = torch.load(cfg["model_path"], map_location=device, weights_only=False)
  model.load_state_dict(state_dict)
  model = model.to(device)
  model.eval()

  test_dataset, _ = get_video_frames(cfg["video_path"], resize=224)
  codec = create_codec(test_dataset, model)

  output_dir       = cfg["output_dir"]
  output_video_dir = os.path.join(output_dir, "decoded_videos")
  os.makedirs(output_video_dir, exist_ok=True)

  colors = plt.cm.viridis(np.linspace(0, 1, 12))[::-1]
  plt.figure(figsize=(7.5, 7.5 / 1.5))

  for used_filter in range(1, 13):
    model.p = used_filter / 12
    sizes, psnrs, ms_ssims = eval_model(
      model, used_filter, test_dataset, codec, output_dir, output_video_dir
    )

    bpps     = [(s * 8 * 1024) / (224 * 224) for s in sizes]
    avg_bpp  = np.mean(bpps)
    avg_psnr = np.mean(psnrs)
    avg_ms   = np.mean(ms_ssims)
    print(f"Filters: {used_filter:2d} | bpp: {avg_bpp:.4f} | "
        f"PSNR: {avg_psnr:.2f} dB | MS-SSIM: {avg_ms:.2f} dB")

    plt.subplot(2, 1, 1)
    plt.plot(bpps, label=f'[0:{used_filter}]', color=colors[used_filter - 1])
    plt.ylabel('bpp'); plt.xticks([]); plt.legend(ncol=4, fontsize=6, loc='lower right')

    plt.subplot(2, 1, 2)
    plt.plot(psnrs,    label=f'{used_filter}', color=colors[used_filter - 1])
    plt.plot(ms_ssims, label=f'{used_filter}', color=colors[used_filter - 1])
    plt.xlabel('Frame Index'); plt.ylabel('PSNR / MS-SSIM (dB)')

  plt.tight_layout()
  plt.savefig('rd_curves.pdf', bbox_inches='tight')
  plt.show()
  print("Evaluation complete. Plot saved to rd_curves.pdf")

In [ ]:
# Uncomment to run video evaluation:
run_video_eval(CONFIG)

# Quick-Start Examples

In [ ]:
# ── Example A: Train from scratch ────────────────────────────────────────────
# CONFIG["train_data_dir"] = "/your/train/images/"
# CONFIG["val_data_dir"]   = "/your/val/images/"
# model = run_training(CONFIG)

# ── Example B: Evaluate a saved model on a video ─────────────────────────────
# CONFIG["model_path"] = "MCUCoder.pth"
# CONFIG["video_path"] = "my_video.mp4"
# run_video_eval(CONFIG)

# ── Example C: Prepare a custom ImageNet subset ──────────────────────────────
# CONFIG["imagenet_raw_dir"]      = "/path/to/imagenet/train/"
# CONFIG["imagenet_out_dir"]      = "/path/to/output/"
# CONFIG["num_images_to_select"]  = 50_000
# prepare_imagenet(CONFIG)